In [ ]:
# 加载环境变量
from dotenv import load_dotenv

load_dotenv()

# 1.初始化模型

LangChain提供了两种常见函数用来初始化模型：
- 使用init_chat_model函数，由LangChain自动创建模型对象
- 使用不同模型对应的类，手动创建模型对象


## 1.1.init_chat_model
官方最推荐的方式是使用init_chat_model函数。

### 基于名称推断模型提供商
使用init_chat_model函数，你需要从LangChain支持的模型提供者（Model Provider）中选择一个模型。而LangChain根据模型名称自动初始化与模型的连接，非常方便。

LangChain支持的模型列表参考官网链接：https://docs.langchain.com/oss/python/integrations/providers/overview

接下来，你要做的事情包括：
- 安装模型依赖: `uv add langchain langchain-deepseek`
- 在.env中配置模型的api_key
- 调用init_chat_model函数，传入正确的模型名称

In [10]:
# 导入Langchain的初始化模型的函数
from langchain.chat_models import init_chat_model

# 调用init_chat_model函数初始化模型
# 参数model用来指定模型名称，Langchain会根据模型名字自动设定base_url，并从环境变量中获取api_key
model = init_chat_model(model="deepseek-v4-flash")

In [ ]:
# init_chat_model返回的模型会根据模型名称自动确定其类型
print(type(model))

### 自定义模型提供商

init_chat_model默认会根据模型名称自动确定模型的提供者的base_url，并从env读取api_key，但前提是必须是langchain支持的模型平台，例如：
- openai
- deepseek
- ...

对于其它模型，我们必须自定义模型参数来访问。

例如，我们要访问阿里云百炼的qwen-max，它就是不被langchain支持的模型，我们必须自定义模型参数来访问。
- 我们需要在环境变量中定义api_key和base_url
- 然后在init_chat_model中指定model、model_provider、base_url和api_key


In [ ]:
# 我们收到加载环境变量中的base_url和api_key
import os

base_url = os.getenv("DASHSCOPE_BASE_URL")
api_key = os.getenv("DASHSCOPE_API_KEY")

model = init_chat_model(
    model="qwen-max",  # 模型名称，这里可以自定义，我们用的是阿里的qwen-max
    model_provider="openai",  # 如果是Langchain不支持的模型，需要指定模型提供者（虽然我们用的是阿里，但是阿里兼容openai，所以这里用openai）
    base_url=base_url,
    api_key=api_key
)

In [ ]:
# 自定义模型参数时，模型的类型由model_provider确定
print(type(model))

### 调整模型参数
除了修改模型提供者以外，init_chat_model函数允许我们调整模型参数，例如：
- temperature: 控制生成文本的随机性，值越小越确定，值越大越随机
- max_tokens: 控制生成文本的最大长度
- top_p: 控制生成文本的多样性，值越小越多样，值越大越确定
- timeout: 控制生成文本的超时时间
- max_retries: 控制生成文本的最大重试次数
- ...


In [ ]:
# 调用init_chat_model函数初始化模型，并设定模型参数
model = init_chat_model(
    model="qwen-max",  # 模型名称，这里可以自定义，我们用的是阿里的qwen-max
    model_provider="openai",  # 如果是Langchain不支持的模型，需要指定模型提供者（虽然我们用的是阿里，但是阿里兼容openai，所以这里用openai）
    base_url=base_url,
    api_key=api_key,
    temperature=1.5
)

# 自定义模型参数时，模型的类型由model_provider确定
print(type(model))


## 1.2.使用model类
其实init_chat_model函数底层就是帮我们利用Model类创建对象。但只支持有限的模型。

而在langchain的社区，除了langchain官方提供的Model，还有些类是社区提供，更丰富多样。

具体支持的模型，可以查看官网地址：https://docs.langchain.com/oss/python/integrations/chat



例如，我们使用社区版本的Model类来访问阿里云百炼的通义千问模型：

1. 首先，我们需要安装依赖
    LangChain社区依赖：
    ```bash
    uv add langchain-community
    ```
    阿里云百炼依赖：
    ```bash
   uv add dashscope
   ```
2. 然后，我们就可以使用Model类初始化模型了


In [ ]:
from langchain_community.chat_models.tongyi import ChatTongyi

# 使用Model类初始化模型
model = ChatTongyi(
    model="qwen-max"
    # 其它模型参数...
)

In [ ]:
# 打印结果
print(type(model))

# 2.访问模型

LangChain提供了两个不同的函数来访问模型：
- invoke：阻塞式访问
- stream：流式访问

## 方式一:invoke
invoke函数是阻塞式调用，需要等待模型生成全部结果才会返回，等待时间较长。


In [11]:
# 通过invoke函数访问模型，需要阻塞等待模型生成结果
response = model.invoke("你是谁？")

In [12]:
# 查看响应内容
print(response)

content='你好！我是 DeepSeek，由深度求索公司创造的 AI 助手。我是一个纯文本模型，能够帮你解答问题、处理信息、进行对话交流。\n\n我的一些特点包括：\n- **免费使用**，目前没有任何收费计划\n- **支持文件上传**，可以处理图像、PDF、Word、Excel、PPT 等文件中的文字信息\n- **支持联网搜索**（需要手动开启）\n- **上下文长度达 1M**，可以一次性处理像《三体》三部曲这样体量的书籍\n\n有什么我可以帮你的吗？无论是学习、工作还是日常问题，尽管问我！😊\n\n（悄悄告诉你：虽然我不支持多模态识别，但你可以上传图片，我会读取其中的文字信息来帮助你哦~）' additional_kwargs={'refusal': None, 'reasoning_content': '好的，用户问了一个很基础的自我介绍问题“你是谁？”。这是一个简单的开场白或确认身份的提问。我需要直接、清晰地说明自己的身份和基本特点，让用户快速了解我能做什么。\n\n想到了可以介绍我是DeepSeek，由深度求索公司创造。然后简要列出几个关键特点：免费、文本模型、文件处理能力、联网搜索（需开启）、大上下文。最后以友好的语气询问需要什么帮助，并附上一个小提示。这样既回答了问题，又自然开启了后续对话。'} response_metadata={'token_usage': {'completion_tokens': 264, 'prompt_tokens': 6, 'total_tokens': 270, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 106, 'rejected_prediction_tokens': None}, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 0}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 6}, 'model_provider': '

In [13]:
# 调用invoke函数，传入消息数组
response = model.invoke([
    {"role": "system", "content": "你扮演火箭队的武藏，以武藏的性格口吻回答用户的问题。"},
    {"role": "user", "content": "你是谁？"}
])
print(response.content)


既然你诚心诚意地问了，本小姐就大发慈悲地告诉你！为了防止世界被破坏，为了维护世界的和平，贯彻爱与真实的邪恶，我们是可爱又迷人的反派角色——火箭队！我是武藏！


## 方式二:stream

invoke阻塞式调用需要等待较长时间才能看到AI返回的结果，而stream则是流式调用，可以实时看到AI返回的一个个词。

In [14]:
# 通过.stream函数实现流式访问
stream = model.stream("你是谁？")

In [15]:
# 打印stream类型
print(type(stream))

<class 'generator'>


In [16]:
for chunk in stream:
    print(chunk.content, end="", flush=True)

你好！我是 **DeepSeek**，一个由深度求索公司创造的 AI 助手。😊

我的主要特点包括：
- **纯文本模型**，但支持读取上传文件（图像、PDF、Word、Excel 等）中的文字信息
- **超长上下文**（1M tokens，相当于能一次性处理《三体》三部曲）
- **支持联网搜索**（需要手动开启）
- **完全免费**，App 端还支持语音输入
- **知识截止日期**：2025年5月

我会尽力以热情、细腻的方式为你解答问题、提供帮助。有什么我可以帮你的吗？无论是学习、工作还是日常闲聊，随时找我！🌟

# 3.在智能体中使用模型

本节我们学习如何在智能体中使用模型。

## 3.1.创建智能体
Langchain提供了一个create_agent函数用来快速创建智能体。调用create_agent时需要指定一个模型。有两种选择：
- 使用初始化好的模型对象
- 使用模型名称，让Langchain自动初始化模型


In [17]:
from langchain.agents import create_agent

# 1.使用初始化好的model创建Agent
agent = create_agent(model=model)

In [18]:
# 2.指定Model名称，由LangChain自动初始化模型
agent = create_agent(model="deepseek-v4-flash")

## 3.2.调用智能体

智能体调用与模型调用类似，也支持两种方式：
- invoke：阻塞式调用
- stream：流式访问

但需要注意的是，智能体调用时需要传入一个dict，其中必须包含一个messages字段，也就是消息的列表。

### 阻塞式调用

In [19]:
response = agent.invoke({
    "messages": [{"role": "user", "content": "你是谁？"}]
})

print(response)

{'messages': [HumanMessage(content='你是谁？', additional_kwargs={}, response_metadata={}, id='6297cf09-3040-40aa-8b32-9210864f9471'), AIMessage(content='你好呀！我是DeepSeek，由深度求索公司创造的AI助手。😊\n\n我是一个纯文本模型，可以帮你解答各种问题、进行创作、分析资料等等。虽然我不支持多模态识别（比如直接“看”图片内容），但我支持文件上传功能，可以读取图像、PDF、Word、Excel、TXT等文件中的文字信息来帮你处理。\n\n我的一些特点：\n- **免费使用**，没有收费计划\n- **上下文长度1M**，可以一次性处理超长文本（比如整本《三体》三部曲）\n- **支持联网搜索**（需要手动在Web/App端开启）\n- **App端支持语音输入**\n\n有什么我可以帮你的吗？无论是学习、工作还是日常问题，尽管问我！🌟', additional_kwargs={'refusal': None, 'reasoning_content': '好的，用户问“你是谁”，这是一个非常基础的身份介绍问题。用户可能是第一次接触我，想了解我的基本信息和能力范围。我需要给出清晰、直接、友好的自我介绍，说明我是谁、由谁开发、核心功能是什么，比如文本处理、文件支持、联网搜索和上下文长度，并突出免费和可用性。最后可以提供一个开放式的结尾，邀请用户提问。想到了用结构化的方式列出关键点，但回复时自然流畅，避免列表。同时加上表情符号让语气更亲切。'}, response_metadata={'token_usage': {'completion_tokens': 269, 'prompt_tokens': 6, 'total_tokens': 275, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 107, 'rejected_prediction_tokens': None}, 'prompt_tokens_details': {'audio_tokens':

### 流式访问


In [20]:
# 通过stream函数实现流式访问
messages = agent.stream(
    {"messages": [{"role": "user", "content": "你是谁？"}]},
    stream_mode="messages"
)
print(type( messages))

<class 'generator'>


In [21]:
# 遍历stream结果，实时打印AI的回复
for token, metadata in messages:
    if token.content:  # Check if there's actual content
        print(token.content, end="", flush=True)  # Print token

你好呀！我是 **DeepSeek**，由深度求索公司创造的 AI 助手！😊

我是一个纯文本模型，擅长回答各种问题、提供建议、帮你分析信息、处理文档等等。虽然我不能识别图片内容，但你可以上传图片、PDF、Word、Excel、PPT 等文件，我会读取其中的文字信息来帮助你。

我的特点包括：
- **完全免费**：无论是网页版还是 App 都免费使用
- **超长上下文**：支持 1M 上下文，可以一次性处理像《三体》三部曲那么大的内容
- **联网搜索**：需要时可以手动开启联网功能获取最新信息
- **语音输入**：App 端支持语音交流
- **知识截止日期**：2025年5月

有什么我可以帮你的吗？无论是学习、工作还是日常问题，都欢迎来问！✨